<a href="https://colab.research.google.com/github/toxzak-svg/hgsel-moe/blob/main/notebooks/phase4_colab_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HGSEL Phase 4: Colab Pro Validation

This notebook runs Phase 4 multi-GPU validation experiments on Google Colab Pro.

**Requirements:**
- Colab Pro subscription
- GPU runtime (preferably A100 or V100)

**What this validates:**
1. GPU baseline benchmark (control condition)
2. Single-GPU HGSEL vs Dense comparison
3. Memory profiling
4. Distributed smoke tests (if 2+ GPUs available)

**Estimated runtime:** 2-4 hours on A100, 4-6 hours on V100/T4

## 1. Setup: Check GPU and Install Dependencies

In [1]:
# Check GPU availability
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"CUDA version: {torch.version.cuda}")

Thu Mar  5 17:12:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone Repository or Upload Code

In [2]:
# Option 1: Clone from GitHub (if repo is public)
# !git clone https://github.com/toxzak-svg/hgsel-moe.git
# %cd hgsel-moe

# Option 2: Upload from Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Adjust path to where you uploaded the code
import os
os.chdir('/content')

# If you zipped and uploaded to Drive:
# !unzip /content/drive/MyDrive/hgsel-moe.zip
# %cd hgsel-moe

# Or use git clone from your local if you pushed to GitHub:
!git clone https://github.com/toxzak-svg/hgsel-moe.git
%cd hgsel-moe

Mounted at /content/drive
Cloning into 'hgsel-moe'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 143 (delta 34), reused 136 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 315.01 KiB | 477.00 KiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/hgsel-moe


In [3]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .

# Verify installation
import hgsel
print(f"HGSEL imported successfully")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hgsel (pyproject.toml) ... done
HGSEL imported successfully


## 3. Phase 4 Experiment 1: GPU Baseline Benchmark

**Goal:** Establish single-GPU control condition

**Metrics:**
- Throughput (tokens/sec)
- Memory usage
- Forward/backward timing

**Expected runtime:** 20-30 minutes

In [ ]:
# Run GPU baseline benchmark
!python experiments/train_gpu_baseline.py \
    --device cuda \
    --models dense,hgsel \
    --batch-size 16 \
    --seq-length 128 \
    --d-model 256 \
    --d-ff 1024 \
    --num-layers 4 \
    --num-experts 64 \
    --warmup-steps 10 \
    --bench-steps 50 \
    --output results/gpu_baseline/colab_baseline.json

[baseline] device=cuda dtype=torch.float32 models=['dense', 'hgsel']
[baseline] batch=16 seq=128 steps=50 warmup=10
[baseline] benchmarking dense ...
[baseline] benchmarking hgsel ...


In [ ]:
# View results
import json
with open('results/gpu_baseline/colab_baseline.json', 'r') as f:
    results = json.load(f)

print("=" * 60)
print("GPU BASELINE RESULTS")
print("=" * 60)

for model_name, metrics in results.items():
    print(f"\n{model_name.upper()}:")
    print(f"  Throughput: {metrics.get('tokens_per_sec', 0):.1f} tokens/sec")
    print(f"  Peak Memory: {metrics.get('peak_memory_mb', 0):.1f} MB")
    print(f"  Avg Step Time: {metrics.get('avg_step_ms', 0):.2f} ms")

# Calculate speedup
if 'dense' in results and 'hgsel' in results:
    speedup = results['hgsel']['tokens_per_sec'] / results['dense']['tokens_per_sec']
    print(f"\nHGSEL Speedup: {speedup:.2f}x")

## 4. Phase 4 Experiment 2: Convergence Test

**Goal:** Verify HGSEL trains correctly on GPU

**Expected runtime:** 30-45 minutes

In [ ]:
# Quick convergence test
!python experiments/phase3_quick_test.py \
    --device cuda \
    --batch-size 16 \
    --num-epochs 3 \
    --checkpoint-dir checkpoints/phase4_colab

## 5. Phase 4 Experiment 3: Token Exchange Microbenchmark

**Goal:** Test distributed token exchange (if multi-GPU available)

**Expected runtime:** 15-20 minutes

In [ ]:
# Check if we have multiple GPUs
num_gpus = torch.cuda.device_count()
print(f"Available GPUs: {num_gpus}")

if num_gpus >= 2:
    print("Running distributed token exchange microbenchmark...")
    !python experiments/benchmark_token_exchange_micro.py \
        --output results/token_exchange_micro/colab_exchange.json
else:
    print("Single GPU detected. Skipping distributed experiments.")
    print("Note: This is OK! Single-GPU validation proves HGSEL works.")
    print("Multi-GPU can be tested elsewhere if needed.")

## 6. Phase 4 Experiment 4: Memory Profiling

**Goal:** Understand memory breakdown

**Expected runtime:** 10-15 minutes

In [ ]:
# Run memory profiling experiment
import sys
sys.path.insert(0, '.')

from experiments.baselines.dense_transformer import TransformerModel
from hgsel.layer import HGSELLayer
from hgsel.distributed.memory_profiler import MemoryProfiler, estimate_model_memory_requirements

# Create models
print("Creating HGSEL model...")
model = TransformerModel(
    vocab_size=256,
    d_model=256,
    d_ff=1024,
    n_layers=4,
    n_heads=4,
    mlp_class=HGSELLayer
).cuda()

# Estimate memory
estimate = estimate_model_memory_requirements(
    model=model,
    batch_size=16,
    seq_length=128,
    dtype=torch.float32
)

print("\n" + "=" * 60)
print("MEMORY ESTIMATE")
print("=" * 60)
for key, value in estimate.items():
    print(f"{key}: {value}")

## 7. Generate Phase 4 Gate Report

**Goal:** Consolidate all results into a single report

In [ ]:
# Check if gate report script exists
import os
if os.path.exists('experiments/phase4_gate_report.py'):
    print("Generating Phase 4 gate report...")
    !python experiments/phase4_gate_report.py \
        --baseline-json results/gpu_baseline/colab_baseline.json \
        --output results/phase4/colab_gate_report.json
else:
    print("Gate report script not found. Skipping...")
    print("Results are saved in individual JSON files.")

## 8. Save Results to Drive

**Important:** Save results before session expires!

In [ ]:
# Copy results to Google Drive
!mkdir -p /content/drive/MyDrive/hgsel_results
!cp -r results/* /content/drive/MyDrive/hgsel_results/
!cp -r checkpoints/phase4_colab /content/drive/MyDrive/hgsel_results/ 2>/dev/null || true

print("✓ Results saved to Google Drive: /MyDrive/hgsel_results/")

## 9. Phase 4 Decision: GO/NO-GO

**Success Criteria:**
- ✅ HGSEL throughput ≥ 0.8x dense (accounting for increased capacity)
- ✅ Memory usage < 80% of GPU VRAM
- ✅ Training converges (loss decreases)
- ✅ Expert utilization balanced (entropy > 0.8)

**If all pass:** Phase 4 validated! Ready for deeper benchmarking or move to Phase 5.

**If any fail:** Debug or reassess architecture before continuing.

In [ ]:
print("Phase 4 Colab validation complete!")
print("\nNext steps:")
print("1. Review results in /content/drive/MyDrive/hgsel_results/")
print("2. Check if success criteria are met")
print("3. If pass: Consider running full Phase 5 benchmarks")
print("4. If fail: Debug specific issues and re-run")